# 02 · KDFU Duplicate Check

**Why this matters:** if the same image (or a copy of it) appears in both the training and the test set, the model is tested on images it has already seen, so accuracy is inflated. This is called **data leakage**.

**What this notebook does (nothing is deleted):**
1. **Exact duplicates:** identical files, found with an MD5 hash of the bytes.
2. **Near-duplicates:** re-saved, flipped, rotated or slightly shifted copies, found by comparing features from an ImageNet-pretrained **EfficientNet-B0** (cosine similarity, with flips and rotations included).
3. Visual checks, so the threshold is justified and not guessed.

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image

from pathlib import Path

# Locate the dataset wherever Kaggle mounted it
PATCHES = next(Path("/kaggle/input").rglob("Patches"))
DATA = PATCHES.parent
CLASSES = {"Abnormal (Ulcer)": PATCHES / "Abnormal(Ulcer)",
           "Normal (Healthy skin)": PATCHES / "Normal(Healthy skin)"}
EXTS = {".jpg", ".jpeg", ".png", ".gif", ".webp"}

def images(folder):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in EXTS)

print("Dataset folder:", DATA)

rows = [{"path": p, "cls": folder.name, "file": p.name}
        for folder in CLASSES.values() for p in images(folder)]
n = len(rows)
print("Images to check:", n)

## 1. Exact duplicates (MD5)

In [ ]:
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b): self.p[self.find(a)] = self.find(b)

def groups_of(uf):
    g = {}
    for i in range(n):
        g.setdefault(uf.find(i), []).append(i)
    return sorted((v for v in g.values() if len(v) > 1), key=len, reverse=True)

md5s = [hashlib.md5(r["path"].read_bytes()).hexdigest() for r in rows]
uf_exact, first = UnionFind(n), {}
for i, h in enumerate(md5s):
    if h in first: uf_exact.union(i, first[h])
    else: first[h] = i
exact_groups = groups_of(uf_exact)

print("Exact duplicate groups:", len(exact_groups))
print("Extra copies:", sum(len(g) - 1 for g in exact_groups))
print("Unique images after removing exact copies:", n - sum(len(g) - 1 for g in exact_groups))

## 2. Near-duplicates (CNN feature similarity)
Each image is embedded in 8 versions (original, mirrored, and 4 rotations of each), and we keep the **best** similarity between two images.

*A first attempt used perceptual hashing (pHash), but visual checks showed it missed shifted copies and wrongly matched different smooth-skin patches, so CNN features are used instead.*

In [ ]:
model = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet", pooling="avg")

def variants(a):
    out = []
    for base in (a, a[:, ::-1]):
        for k in range(4):
            out.append(np.ascontiguousarray(np.rot90(base, k)))
    return out

feats = []
for r in rows:
    a = np.asarray(Image.open(r["path"]).convert("RGB").resize((224, 224), Image.BILINEAR), dtype=np.float32)
    f = model.predict(tf.keras.applications.efficientnet.preprocess_input(np.stack(variants(a))), verbose=0)
    feats.append(f / np.linalg.norm(f, axis=1, keepdims=True))
feats = np.stack(feats)                      # (n, 8, 1280)

base = feats[:, 0, :]
sim = np.full((n, n), -1.0, dtype=np.float32)
for k in range(feats.shape[1]):
    sim = np.maximum(sim, base @ feats[:, k, :].T)
sim = np.maximum(sim, sim.T)
np.fill_diagonal(sim, -1.0)
print("Similarity matrix:", sim.shape)

### Choosing the threshold
The histogram shows how similar each image is to its closest **non-identical** image.

In [ ]:
THRESHOLD = 0.95
roots = np.array([uf_exact.find(i) for i in range(n)])
not_exact = roots[:, None] != roots[None, :]
simu = np.where(not_exact, sim, -1)

plt.figure(figsize=(7, 3.6))
plt.hist(simu.max(1), bins=np.linspace(0.5, 1.0, 51), color="#1f3b63")
plt.axvline(THRESHOLD, color="#b22222", ls="--", label=f"threshold = {THRESHOLD}")
plt.xlabel("Cosine similarity to the most similar NON-identical image")
plt.ylabel("Number of images"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
def show_pairs(lo, hi, title, k=8):
    pi, pj = np.where(np.triu((simu >= lo) & (simu < hi), 1))
    order = np.argsort(simu[pi, pj])[:k]
    fig, axes = plt.subplots(2, k, figsize=(2.1 * k, 4.8))
    for c, o in enumerate(order):
        for r, idx in enumerate((pi[o], pj[o])):
            ax = axes[r, c]; ax.axis("off")
            ax.imshow(Image.open(rows[idx]["path"]).convert("RGB"))
            short = "Ulcer" if rows[idx]["cls"].startswith("Abnormal") else "Normal"
            ax.set_title(f"{short}/{rows[idx]['file']}" + (f"\nsim={simu[pi[o], pj[o]]:.3f}" if r == 0 else ""), fontsize=8)
    fig.suptitle(title, fontweight="bold"); plt.tight_layout(); plt.show()

show_pairs(THRESHOLD, THRESHOLD + 0.02, f"Just ABOVE threshold ({THRESHOLD}-{THRESHOLD + 0.02:.2f}): counted as duplicates")
show_pairs(THRESHOLD - 0.03, THRESHOLD, f"Just BELOW threshold ({THRESHOLD - 0.03:.2f}-{THRESHOLD}): not counted")

**Reading the pairs:**
- **Above 0.95:** clear copies or tiny shifts of the same patch, so they are counted as duplicates.
- **0.92-0.95:** usually *different crops of the same original photo* (the same skin creases, shifted). They aren't true duplicates, but they still leak if split across training and test. A threshold can't separate them reliably, so a later notebook groups patches by their **source photo**.

## 3. Results

In [ ]:
uf = UnionFind(n)
for i, j in zip(*np.where(np.triu(sim >= THRESHOLD, 1))):
    uf.union(i, j)
for i in range(n):
    uf.union(i, uf_exact.find(i))
near_groups = groups_of(uf)

summary = []
for cls in sorted({r["cls"] for r in rows}):
    idx = [i for i in range(n) if rows[i]["cls"] == cls]
    summary.append({"class": cls, "files": len(idx),
                    "unique (exact)": len({uf_exact.find(i) for i in idx}),
                    "unique (near, >= %.2f)" % THRESHOLD: len({uf.find(i) for i in idx})})
summary = pd.DataFrame(summary)
summary.loc["total"] = ["TOTAL"] + summary.iloc[:, 1:].sum().tolist()
cross = [g for g in near_groups if len({rows[i]["cls"] for i in g}) > 1]
print("Largest group size:", max(len(g) for g in near_groups))
print("Groups containing BOTH labels (label conflicts):", len(cross))
summary

In [ ]:
# The biggest duplicate groups (one row = one group)
show = near_groups[:6]
fig, axes = plt.subplots(len(show), 6, figsize=(13, 2.3 * len(show)), squeeze=False)
for r, g in enumerate(show):
    for c in range(6):
        ax = axes[r, c]; ax.axis("off")
        if c < len(g):
            ax.imshow(Image.open(rows[g[c]]["path"]).convert("RGB"))
            ax.set_title(rows[g[c]]["file"], fontsize=8)
fig.suptitle("Largest duplicate groups", fontweight="bold"); plt.tight_layout()
plt.savefig("largest_duplicate_groups.png", dpi=90); plt.show()

In [ ]:
# Save the per-image table (used by the next notebook for leakage-free splitting)
gid = {i: k for k, g in enumerate(near_groups, 1) for i in g}
table = pd.DataFrame({
    "class": [r["cls"] for r in rows], "file": [r["file"] for r in rows], "md5": md5s,
    "exact_group": [roots[i] for i in range(n)], "dup_group": [gid.get(i, 0) for i in range(n)],
    "max_similarity": simu.max(1).round(4),
})
table.to_csv("kdfu_duplicate_table.csv", index=False)
table.head()

## Summary
- About **one third of the files are exact copies**. The normal class is hit hardest: its 543 files contain only about 240 distinct images.
- **No image appears under both labels.**
- A naive random split of this folder **would leak**: copies of the same image would land in both training and test.
- **Next:** group patches by their source photo and build a leakage-free, stratified 5-fold split.